In [0]:
from pyspark.sql import functions as F

df_loan = spark.read.parquet(
    "/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Loan"
)

duplicate_loans = df_loan.groupBy("LoanID").agg(
    F.count("LoanID").alias("count")
).filter("count >1")

duplicate_loans.cache().count().display()


In [0]:
client_df = spark.read.parquet("/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Client")
loan_df = spark.read.parquet("/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Loan")
accountTrx_df = spark.read.parquet("/Volumes/bfsi_lakehouse/raw/synthetic_data/t_AccountTrx")

client_df.createOrReplaceTempView("t_Client")
loan_df.createOrReplaceTempView("t_Loan")
accountTrx_df.createOrReplaceTempView("t_AccountTrx")

spark.sql("""
        WITH t_Client AS
        (
          SELECT 
                't_Client' AS TableName,
                dt AS DateFrom, 
                count(*) AS Records
          FROM t_Client
          GROUP BY dt
        ),
        t_Loan AS
        (  SELECT 
                't_Loan' AS TableName,
                dt AS DateFrom, 
                count(*) AS Records
          FROM t_Loan
          GROUP BY dt
        ),
        t_AccountTrx AS
        (
          SELECT 
                't_AccountTrx' AS TableName,
                dt AS DateFrom, 
                count(*) AS Records
          FROM t_AccountTrx
          GROUP BY dt
         )
         SELECT * FROM t_Client
         UNION ALL
         SELECT * FROM t_Loan
         UNION ALL
         SELECT * FROM t_AccountTrx
         ORDER BY TableName,DateFrom
      """).show()



In [0]:
spark.sql("""
          INSERT INTO bfsi_lakehouse.metadata.etl_process_log
          (log_id,run_id,table_id,table_name,process_type,load_type,run_dt,status,started_at)
          SELECT
            UUID() AS log_id,
            UUID() as run_id,-- (SELECT MAX(run_id) FROM etl_process_master) or it can be a veriable, which will be calclulate before every run
            1 AS table_id, -- later table id from table_config
            't_client' AS table_name,
            'BRONZE' AS process_type,
            'INCREMENTAL' AS load_type,
            CURRENT_DATE() AS run_dt,
            'SUCCESS' AS status,
            CURRENT_TIMESTAMP() AS started_at
          """)

spark.sql("""SELECT * FROM bfsi_lakehouse.metadata.etl_process_log""").show()

In [0]:
spark.sql("""describe bfsi_lakehouse.metadata.etl_process_log""").show()

In [0]:
from datetime import datetime
import uuid
from pyspark.sql import functions as F

row = [(
    int('10'),  # log_id
    int('10'),  # run_id
    int('11'),  # table_id
    't_Client',         # table_name
    'BRONZE',           # process_type
    'INCREMENTAL',      # load_type
    datetime.now().date(),# run_dt
    "STARTED",          # status
    None,               # rows_read
    None,               # rows_written
    None,               # delta_version_before
    None,               # delta_version_after
    datetime.now(),     # started_at
    datetime.now(),     # finished_at
    None,               # duration_seconds
    None,               # error_message
    None                # error_stacktrace
)]

# 1. Fetch the exact schema from your target table
target_schema = spark.table("bfsi_lakehouse.metadata.etl_process_log").schema

# 2. Pass that schema into createDataFrame
spark.createDataFrame(row, schema=target_schema).write \
    .mode("append") \
    .saveAsTable("bfsi_lakehouse.metadata.etl_process_log")

spark.sql("""SELECT * FROM bfsi_lakehouse.metadata.etl_process_log""").show()

In [0]:
run_id = 10
spark.sql(f"""
    UPDATE bfsi_lakehouse.metadata.etl_process_log
    SET status = 'SUCCESS',
        finished_at = current_timestamp(),
        rows_written = 98765
    WHERE run_id = '{run_id}'
""")

In [0]:
spark.sql("""SELECT * FROM bfsi_lakehouse.metadata.etl_process_log""").show()

In [0]:
spark.sql("DESCRIBE HISTORY bfsi_lakehouse.metadata.etl_process_log").show(10, False)

In [0]:
# 4.5 - time travel
spark.sql("SELECT COUNT(*) FROM bfsi_lakehouse.metadata.etl_process_log VERSION AS OF 0").show()
spark.sql("SELECT COUNT(*) FROM bfsi_lakehouse.metadata.etl_process_log").show()

In [0]:
# 4.6 - cleanup fake rows
spark.sql("""
    DELETE FROM bfsi_lakehouse.metadata.etl_process_log
    WHERE table_name = 't_Client'
    AND log_id IN ('10', '1')
""")